# پروژه ارشد: DMD / EDMD روی داده‌های فارکس (EUR/USD)
### نسخه‌ی نهایی — افق پیش‌بینی ۲ روزه، تحلیل کامل، سازگار با نسخه‌ی src/dmd_indicator.py

این نوت‌بوک برای **تحقیق و تحلیل تعاملی** است. برای اجرای خودکار روزانه از
`src/dmd_indicator.py` + `run_daily.bat` استفاده کنید (به README.md مراجعه کنید).


## ۱. نصب و ایمپورت

In [ ]:
# فقط بار اول لازم است:
# !pip install yfinance pandas numpy matplotlib


In [ ]:
import sys, os
sys.path.append(os.path.abspath("."))  # برای import کردن dmd_edmd.py از همین پوشه‌ی src

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from dmd_edmd import hankel_embed, DMD, EDMD

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True


## ۲. دریافت داده

نمادهای پیشنهادی فارکس در yfinance: `EURUSD=X`, `GBPUSD=X`, `USDJPY=X`


In [ ]:
SYMBOL = "EURUSD=X"
START = "2018-01-01"
END = "2025-01-01"
INTERVAL = "1d"
HORIZON = 2   # افق پیش‌بینی: ۲ روز آینده

try:
    raw = yf.download(SYMBOL, start=START, end=END, interval=INTERVAL, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    raw = raw[["Open", "High", "Low", "Close", "Volume"]].dropna()
    raw.index.name = "datetime"
    for col in ["Open", "High", "Low", "Close", "Volume"]:
        raw[col] = np.asarray(raw[col]).reshape(-1)
    raw.to_csv("../data/raw_data.csv")
    print("دریافت شد. تعداد ردیف:", len(raw))
except Exception as e:
    print("خطا در دریافت آنلاین:", e)
    raw = pd.read_csv("../data/raw_data.csv", index_col="datetime", parse_dates=True)
    print("از فایل محلی خوانده شد. تعداد ردیف:", len(raw))

raw.head()


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
close_vals = np.asarray(raw["Close"]).reshape(-1)
ax[0].plot(raw.index, close_vals)
ax[0].set_title(f"{SYMBOL} - قیمت Close")
log_ret_full = np.diff(np.log(close_vals))
ax[1].plot(raw.index[1:], log_ret_full)
ax[1].set_title("بازده لگاریتمی روزانه")
plt.tight_layout()
plt.show()


## ۳. تفکیک Train / Test

In [ ]:
TEST_SIZE = 250
close = np.asarray(raw["Close"]).reshape(-1)
log_ret = np.diff(np.log(close))
train_series = log_ret[:-TEST_SIZE]
test_series  = log_ret[-TEST_SIZE:]
print("طول train:", len(train_series), " | طول test:", len(test_series))


## ۴. تست صحت پیاده‌سازی

In [ ]:
t = np.linspace(0, 20, 400)
clean_signal = np.sin(2*np.pi*0.3*t) + 0.3*np.sin(2*np.pi*1.1*t)
sig_train = clean_signal[:350]
sig_true_future = clean_signal[350:352]  # ۲ گام - هماهنگ با HORIZON پروژه

Xs = hankel_embed(sig_train, delay=15)
dmd_s = DMD(rank=6).fit(Xs)
pred_future = dmd_s.forecast(Xs[:, -1], n_steps=2)

print("RMSE پیش‌بینی ۲ گام آینده روی سیگنال بدون نویز:", np.sqrt(np.mean((pred_future - sig_true_future)**2)))
print("واقعی:", sig_true_future, " | پیش‌بینی:", pred_future)


## ۵. انتخاب پارامتر (Delay و Rank)

In [ ]:
def one_step_eval(series, delay, rank, val_size=100):
    fit_part = series[: -val_size]
    val_part = series[-val_size:]
    X = hankel_embed(fit_part, delay=delay)
    model = DMD(rank=rank).fit(X)
    preds, history = [], list(fit_part)
    x0 = X[:, -1].copy()
    for true_val in val_part:
        p = model.forecast(x0, n_steps=1)[0]
        preds.append(p)
        history.append(true_val)
        x0 = np.array(history[-delay:])
    return np.sqrt(np.mean((np.array(preds) - val_part) ** 2))

results = []
for delay in [10, 20, 30]:
    for rank in [5, 10, 15]:
        try:
            results.append((delay, rank, one_step_eval(train_series, delay, rank)))
        except Exception:
            results.append((delay, rank, np.nan))

res_df = pd.DataFrame(results, columns=["delay", "rank", "val_rmse"]).sort_values("val_rmse")
res_df


In [ ]:
BEST_DELAY, BEST_RANK, _ = res_df.iloc[0]
BEST_DELAY, BEST_RANK = int(BEST_DELAY), int(BEST_RANK)
print(f"بهترین پارامترها: delay={BEST_DELAY}, rank={BEST_RANK}")


## ۶. ارزیابی نهایی روی Test (افق ۲ روزه، one-step rolling)

In [ ]:
def rolling_forecast_test(train_hist, test_part, delay, rank):
    history = list(train_hist)
    preds = []
    for true_val in test_part:
        X = hankel_embed(np.array(history[-(delay*8):]), delay=delay)
        model = DMD(rank=rank).fit(X)
        p = model.forecast(X[:, -1], n_steps=1)[0]
        preds.append(p)
        history.append(true_val)
    return np.array(preds)

dmd_preds = rolling_forecast_test(train_series, test_series, BEST_DELAY, BEST_RANK)


In [ ]:
def rmse(a, b): return np.sqrt(np.mean((a - b) ** 2))
def mae(a, b):  return np.mean(np.abs(a - b))
def directional_accuracy(pred, true): return np.mean(np.sign(pred) == np.sign(true)) * 100

rw_preds = np.zeros_like(test_series)
mean_preds = np.full_like(test_series, train_series.mean())

summary = pd.DataFrame({
    "مدل": ["DMD", "Random Walk", "میانگین تاریخی"],
    "RMSE": [rmse(dmd_preds, test_series), rmse(rw_preds, test_series), rmse(mean_preds, test_series)],
    "MAE":  [mae(dmd_preds, test_series),  mae(rw_preds, test_series),  mae(mean_preds, test_series)],
    "دقت جهت‌گیری (%)": [directional_accuracy(dmd_preds, test_series),
                          directional_accuracy(rw_preds, test_series),
                          directional_accuracy(mean_preds, test_series)],
})
summary


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(test_series, label="واقعی", alpha=0.8)
plt.plot(dmd_preds, label="پیش‌بینی DMD", alpha=0.8)
plt.axhline(0, color="gray", linestyle=":", linewidth=1)
plt.legend()
plt.title("پیش‌بینی one-step-ahead روی Test")
plt.show()


## ۷. تحلیل مودال با EDMD

In [ ]:
X_full_train = hankel_embed(train_series, delay=BEST_DELAY)
edmd_model = EDMD(rank=min(20, X_full_train.shape[0]*2)).fit(X_full_train)
growth, freq = edmd_model.dominant_frequencies(dt=1.0)
modal_df = pd.DataFrame({"frequency": freq, "growth_rate": growth, "|eigval|": np.abs(edmd_model.eigvals)})
modal_df.sort_values("|eigval|", ascending=False).reset_index(drop=True).head(10)


In [ ]:
plt.figure(figsize=(6, 6))
theta = np.linspace(0, 2*np.pi, 200)
plt.plot(np.cos(theta), np.sin(theta), "k--", linewidth=1, label="دایره واحد")
plt.scatter(edmd_model.eigvals.real, edmd_model.eigvals.imag, c="crimson")
plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(0, color="gray", linewidth=0.5)
plt.title("مقادیر ویژه‌ی EDMD")
plt.legend(); plt.axis("equal")
plt.show()


## ۸. پیش‌بینی نهایی: ۲ روز واقعی آینده (سازگار با اندیکاتور خودکار)

In [ ]:
N_FUTURE = HORIZON  # = 2

full_history = np.concatenate([train_series, test_series])
X_final = hankel_embed(full_history[-(BEST_DELAY*8):], delay=BEST_DELAY)
final_model = DMD(rank=BEST_RANK).fit(X_final)

future_log_returns = final_model.forecast(X_final[:, -1], n_steps=N_FUTURE)
last_price = close[-1]
future_prices = last_price * np.exp(np.cumsum(future_log_returns))

future_dates = pd.bdate_range(raw.index[-1], periods=N_FUTURE + 1)[1:]
forecast_df = pd.DataFrame({
    "تاریخ": future_dates,
    "بازده لگاریتمی پیش‌بینی‌شده": future_log_returns,
    "قیمت پیش‌بینی‌شده": future_prices,
})
forecast_df


In [ ]:
plt.figure(figsize=(12, 4))
history_window = 40
plt.plot(raw.index[-history_window:], close[-history_window:], label="قیمت واقعی (اخیر)")
plt.plot(forecast_df["تاریخ"], forecast_df["قیمت پیش‌بینی‌شده"], "o--", color="crimson", label=f"پیش‌بینی {N_FUTURE} روز آینده")
plt.axvline(raw.index[-1], color="gray", linestyle=":")
plt.legend(); plt.title(f"{SYMBOL} - پیش‌بینی نهایی")
plt.xticks(rotation=30); plt.tight_layout()
plt.show()


## ۹. جمع‌بندی

- این نوت‌بوک برای **تحقیق تعاملی** است (انتخاب پارامتر، ارزیابی، تحلیل مودال).
- برای **اجرای خودکار روزانه به‌عنوان اندیکاتور**، از `dmd_indicator.py` در همین پوشه استفاده کنید
  (منطق مشابه اما ساده‌شده و خودکار، با ثبت لاگ در `data/signals_log.csv`).
- افق پیش‌بینی هر دو فایل الان **۲ روز** است (`HORIZON = 2`).
